# AMORPH AutoTrainer v0.1 — бесплатный облачный запуск

Этот ноутбук проверяет pipeline классификации и извлечения признаков. Базовая демонстрация использует **искусственные данные**: она проверяет корректность установки, обучения, сохранения, предсказаний и Quality Gate, но **не доказывает качество на реальной задаче**.

**Бюджет:** платные API не используются; GPU не требуется. Бесплатность и доступность Colab не гарантируются самой платформой. Установка зависимостей требует интернет-доступа.

1. Скачай `AMORPH_AutoTrainer_v0.1.zip`.
2. Запусти ячейки по порядку. В первой загрузится ZIP с кодом.
3. Чтобы обучать на своих данных позже, подготовь CSV с колонками `text,label` и запусти необязательную ячейку для своих данных.

AutoTrainer v0.1 поддерживает TF-IDF + Logistic Regression и, опционально, embeddings E5. RAG, LoRA/QLoRA и дистилляция ещё не реализованы в этой версии.

In [ ]:
from google.colab import files
from pathlib import Path
from zipfile import ZipFile
import shutil, os

uploaded = files.upload()  # Выбери AMORPH_AutoTrainer_v0.1.zip
zip_name = next((name for name in uploaded if name.lower().endswith('.zip')), None)
if not zip_name:
    raise ValueError('Загрузи ZIP-архив AMORPH_AutoTrainer_v0.1.zip')

extract_dir = Path('/content/amorph_upload')
if extract_dir.exists():
    shutil.rmtree(extract_dir)
extract_dir.mkdir(parents=True)
with ZipFile(zip_name) as archive:
    # Архив — созданный проект AMORPH; проверяем пути перед распаковкой.
    for member in archive.infolist():
        target = (extract_dir / member.filename).resolve()
        if not target.is_relative_to(extract_dir.resolve()):
            raise ValueError('Небезопасный путь в ZIP: ' + member.filename)
    archive.extractall(extract_dir)

candidates = list(extract_dir.rglob('pyproject.toml'))
if not candidates:
    raise FileNotFoundError('В ZIP не найден pyproject.toml')
PROJECT_ROOT = candidates[0].parent
os.chdir(PROJECT_ROOT)
print('Папка проекта:', PROJECT_ROOT)
print('README:', (PROJECT_ROOT / 'README_RU.md').exists())

In [ ]:
import subprocess, sys

# Установка проекта и pytest. При сбое сети не переключаемся на платный ресурс.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.[test]'], check=True)
print('Установка завершена')

In [ ]:
# Проверки: unit-тесты и компиляция Python-кода.
subprocess.run([sys.executable, '-m', 'pytest', '-q'], cwd=PROJECT_ROOT, check=True)
subprocess.run([sys.executable, '-m', 'compileall', '-q', 'amorph_autotrainer'], cwd=PROJECT_ROOT, check=True)
print('Unit-тесты и compileall пройдены')

In [ ]:
# Создаём синтетические данные и запускаем один дешёвый TF-IDF baseline.
# Результат используется только для smoke test.
from pathlib import Path
import subprocess, sys, json, time

WORK = Path('/content/amorph_work')
if WORK.exists():
    shutil.rmtree(WORK)
WORK.mkdir(parents=True)
data_csv = WORK / 'synthetic_demo.csv'
out_dir = WORK / 'runs'
subprocess.run([sys.executable, '-m', 'amorph_autotrainer.cli', 'demo', '--output', str(data_csv), '--examples-per-class', '50'], cwd=PROJECT_ROOT, check=True)
start = time.perf_counter()
result = subprocess.run([sys.executable, '-m', 'amorph_autotrainer.cli', 'train', str(data_csv), '--text-column', 'text', '--label-column', 'label', '--out', str(out_dir), '--strategies', 'tfidf', '--max-runtime-seconds', '600', '--max-candidates', '1', '--bootstrap-samples', '300', '--min-gate-examples-per-class', '5', '--expected-queries', '1000'], cwd=PROJECT_ROOT, text=True, capture_output=True)
print(result.stdout[-12000:])
if result.stderr:
    print('STDERR:', result.stderr[-4000:])
print('Exit code:', result.returncode, '| Wall time:', round(time.perf_counter()-start, 2), 'sec')
if result.returncode not in (0, 2, 3, 4):
    raise RuntimeError('Pipeline failed; see output above')
run_dirs = sorted([p for p in out_dir.iterdir() if p.is_dir() and p.name[:4].isdigit()])
RUN_DIR = run_dirs[-1]
REPORT = json.loads((RUN_DIR / 'report.json').read_text(encoding='utf-8'))
print('Selected:', REPORT['strategy'])
print('Status:', REPORT['status'])
print('Final test metrics (synthetic only):', REPORT['final_test_metrics'])
print('Test used for selection:', REPORT['test_split_used_for_selection'])
print('TCO estimate:', REPORT.get('full_selection_run_tco'))
assert REPORT['test_split_used_for_selection'] is False
assert REPORT['acceptance_gate_sufficient'] is True
print('Пайплайн технически отработал. Метрики синтетических данных не являются доказательством качества на реальных данных.')

In [ ]:
# Проверяем предсказание и извлечение признаков готовой модели.
import pandas as pd
sample_csv = WORK / 'new_texts.csv'
pd.DataFrame({'text': [
    'Команда выиграла матч и вышла в финал турнира.',
    'Исследователи улучшили алгоритм машинного обучения.',
    'Правительство представило программу регионального развития.'
]}).to_csv(sample_csv, index=False)
pred_csv = WORK / 'predictions.csv'
subprocess.run([sys.executable, '-m', 'amorph_autotrainer.cli', 'predict', '--model-dir', str(RUN_DIR / 'selected'), '--input', str(sample_csv), '--output', str(pred_csv)], cwd=PROJECT_ROOT, check=True)
display(pd.read_csv(pred_csv))
features_path = WORK / 'features.npz'
subprocess.run([sys.executable, '-m', 'amorph_autotrainer.cli', 'features', '--model-dir', str(RUN_DIR / 'selected'), '--input', str(sample_csv), '--output', str(features_path)], cwd=PROJECT_ROOT, check=True)
print('Файл признаков:', features_path, '| размер:', features_path.stat().st_size, 'bytes')

## Необязательно: обучить на собственных данных

Когда появятся реальные данные, подготовь CSV с двумя колонками: `text` и `label`. Используй отдельный тестовый набор, если он уже есть; для текущего MVP pipeline сам делит CSV на train, validation, acceptance gate и test. Для данных, сгруппированных по пользователю, источнику или времени, нужна отдельная стратегия split.

Не используй синтетический набор как подтверждение качества. Порог `--min-macro-f1` следует задавать на основании требований реальной задачи. Если набор содержит персональные или конфиденциальные данные, не загружай его в публичный облачный notebook без разрешения и проверки условий обработки данных.

In [ ]:
# Запускай ячейку только когда у тебя есть разрешённый для облачной обработки CSV.
uploaded_data = files.upload()  # выбери CSV с колонками text,label
csv_name = next((name for name in uploaded_data if name.lower().endswith('.csv')), None)
if not csv_name:
    raise ValueError('Не загружен CSV')

real_out = WORK / 'real_runs'
real_result = subprocess.run([
    sys.executable, '-m', 'amorph_autotrainer.cli', 'train', csv_name,
    '--text-column', 'text', '--label-column', 'label',
    '--out', str(real_out), '--strategies', 'tfidf',
    '--max-runtime-seconds', '600', '--max-candidates', '1',
    '--bootstrap-samples', '500', '--min-gate-examples-per-class', '5'
], cwd=PROJECT_ROOT, text=True, capture_output=True)
print(real_result.stdout[-12000:])
if real_result.stderr:
    print('STDERR:', real_result.stderr[-4000:])
if real_result.returncode not in (0, 2, 3, 4):
    raise RuntimeError('Обучение завершилось с технической ошибкой')
print('Проверь report.json, статус Quality Gate и независимый test перед использованием модели.')

## Сохранить результат

После успешного запуска можно скачать папку экспериментов: в ней хранятся модель, отчёт, manifest и SQLite-журнал. Не публикуй артефакты с пользовательскими данными или эмбеддингами без проверки на чувствительную информацию.

In [ ]:
from google.colab import files
artifact_zip = shutil.make_archive('/content/AMORPH_demo_artifacts', 'zip', root_dir=WORK)
print('Подготовлен архив:', artifact_zip)
files.download(artifact_zip)